In [8]:
import os

import clip
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm

torch.random.manual_seed(42)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

TRAIN_ROOT = "data/train"
TEST_DIR = "data/test"
CLASSES = {"dirty": 0, "cleaned": 1}

model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

print(f"Device: {device}")
print(f"Model: ViT-B/32, params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Device: cuda
Model: ViT-B/32, params: 151.3M


In [ ]:
dirty_prompts = [
    "a photo of a dirty plate",
    "a photo of a dirty plate with leftover sauce",
    "a photo of a plate with food residue and stains",
    "a dirty, unwashed plate with scraps on it",
    "a messy plate that has not been cleaned",
    "a used plate covered in grease and crumbs",
]

clean_prompts = [
    "a photo of a clean plate",
    "a photo of a clean, washed plate",
    "a photo of a spotless, empty plate",
    "a freshly washed, shiny plate with no food on it",
    "a sparkling clean dish on a table",
    "a pristine white plate, completely clean",
]


def encode_prompts(prompts):
    """Encode multiple prompts and return their L2-normalised mean embedding."""
    tokens = clip.tokenize(prompts).to(device)
    with torch.no_grad():
        feats = model.encode_text(tokens)
        feats /= feats.norm(dim=-1, keepdim=True)
    mean = feats.mean(dim=0, keepdim=True)
    return mean / mean.norm(dim=-1, keepdim=True)


# Pre-compute ensembled text embeddings (once, not per image)
text_features = torch.cat([
    encode_prompts(dirty_prompts),
    encode_prompts(clean_prompts),
], dim=0)  # (2, D)

# Inference
test_files = sorted(os.listdir(TEST_DIR))
similarities = []

for fn in tqdm(test_files, desc="Zero-shot inference"):
    img = preprocess(Image.open(os.path.join(TEST_DIR, fn))).unsqueeze(0).to(device)
    with torch.no_grad():
        img_feat = model.encode_image(img)
        img_feat /= img_feat.norm(dim=-1, keepdim=True)
        sim = (100.0 * img_feat @ text_features.T).softmax(dim=-1)
    similarities.append(sim.cpu().numpy()[0])

similarities = np.array(similarities)
zs_labels = np.where(similarities[:, 0] - similarities[:, 1] > -0.1, 0, 1)

print(f"Zero-shot — dirty: {(zs_labels == 0).sum()}, cleaned: {(zs_labels == 1).sum()}")

Zero-shot inference: 100%|██████████| 744/744 [00:08<00:00, 85.32it/s]

Zero-shot — dirty: 315, cleaned: 429


In [ ]:
id_to_label = {0: "dirty", 1: "cleaned"}

zs_df = pd.DataFrame({
    "id": [fn[:fn.rfind(".")] for fn in test_files],
    "label": [id_to_label[l] for l in zs_labels],
})

zs_df.to_csv("data/zeroshot_predictions.csv", index=False)
zs_df.head()

,id,label
0,0000,dirty
1,0001,dirty
2,0002,dirty
3,0003,dirty
4,0004,dirty


In [ ]:
def extract_embeddings(image_paths):
    """Return L2-normalised CLIP image embeddings as a numpy array."""
    feats = []
    for p in tqdm(image_paths, desc="Extracting embeddings"):
        img = preprocess(Image.open(p)).unsqueeze(0).to(device)
        with torch.no_grad():
            f = model.encode_image(img)
            f /= f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu().numpy()[0])
    return np.stack(feats)


# --- 1. Build training set ---
train_paths, train_labels = [], []
for cls_name, y_id in CLASSES.items():
    cls_dir = os.path.join(TRAIN_ROOT, cls_name)
    for fn in sorted(os.listdir(cls_dir)):
        if fn.lower().endswith((".jpg", ".jpeg", ".png")):
            train_paths.append(os.path.join(cls_dir, fn))
            train_labels.append(y_id)

X_train = extract_embeddings(train_paths)
y_train = np.array(train_labels)
print(f"Train set: {len(y_train)} images ({(y_train == 0).sum()} dirty, {(y_train == 1).sum()} cleaned)")

# --- 2. Fit logistic regression ---
clf = LogisticRegression(max_iter=2000).fit(X_train, y_train)

# --- 3. Predict on test ---
test_paths = [os.path.join(TEST_DIR, fn) for fn in test_files]
X_test = extract_embeddings(test_paths)

lp_proba = clf.predict_proba(X_test)[:, 1]
lp_labels = (lp_proba >= 0.55).astype(int)

print(f"Linear probe — dirty: {(lp_labels == 0).sum()}, cleaned: {(lp_labels == 1).sum()}")

Extracting embeddings: 100%|██████████| 40/40 [00:00<00:00, 71.96it/s]


Train set: 40 images (20 dirty, 20 cleaned)


Extracting embeddings: 100%|██████████| 744/744 [00:10<00:00, 71.35it/s]

Linear probe — dirty: 466, cleaned: 278


In [5]:
lp_df = pd.DataFrame({
    "id": [fn[:fn.rfind(".")] for fn in test_files],
    "label": [id_to_label[l] for l in lp_labels],
})

lp_df.to_csv("data/linear_probe_predictions.csv", index=False)
lp_df.head()

,id,label
0,0000,dirty
1,0001,dirty
2,0002,cleaned
3,0003,dirty
4,0004,dirty
